In [8]:
from halib import *
from halib.filetype import csvfile
workdir = "./zout/perf/__analyze"
os.makedirs(workdir, exist_ok=True)

dirs = fs.list_dirs(workdir)
ls_dirs = [
    os.path.join(workdir, d) for d in dirs if os.path.isdir(os.path.join(workdir, d))
]

ls_dirs = [d for d in ls_dirs if "__ds_DFire__mt" in d]
assert len(ls_dirs) == 2, f"Expected 2 dirs, got {len(ls_dirs)}"

if "mt_no_temp" in ls_dirs[0]:
    dir_no_temp = ls_dirs[0]
    dir_temp = ls_dirs[1]
else:
    dir_no_temp = ls_dirs[1]
    dir_temp = ls_dirs[0]

csv1_notemp = os.path.join(dir_no_temp, "[per_video]_raw_metric_src_.csv")
csv2_temp = os.path.join(dir_temp, "[per_video]_raw_metric_src_.csv")

df1 = pd.read_csv(csv1_notemp, sep=';', encoding='utf-8')
df2 = pd.read_csv(csv2_temp, sep=';', encoding='utf-8')
df1.drop_duplicates(inplace=True)
df2.drop_duplicates(inplace=True)
df1.rename(columns={"pred": "pred_no_temp", "correct": "correct_no_temp"}, inplace=True)
df2.rename(columns={"pred": "pred_temp", "correct": "correct_temp"}, inplace=True)

df = pd.merge(df1, df2, on=["video_name", "gt"])
cols = ["video_name", "gt", "pred_no_temp", "pred_temp", "correct_no_temp", "correct_temp"]
df = df[cols]

df["diff"] = (df['correct_no_temp'] - df['correct_temp'])
df["correct_both"] = df.apply(lambda row: row['correct_no_temp'] and row['correct_temp'], axis=1)
df.sort_values(by=["correct_both"], ascending=False, inplace=True)
df.to_csv(os.path.join(workdir, "__compare.csv"), sep=';', encoding='utf-8', index=False)

df1 = df[df["correct_both"] == 0]
df1 = df1.sort_values(by=["diff", 'video_name'], ascending=[False, True])
df1.to_csv(
    os.path.join(workdir, "__compare_wrong.csv"),
    sep=";",
    encoding="utf-8",
    index=False,
)

In [3]:
import subprocess

def vstack(video1, video2, output="output.mp4"):
    """
    Stick two videos horizontally using ffmpeg.

    Args:
        video1 (str): Path to the first video.
        video2 (str): Path to the second video.
        output (str): Output file path.
    """
    command = [
        "ffmpeg",
        "-i",
        video1,
        "-i",
        video2,
        "-filter_complex",
        "hstack=inputs=2",
        "-c:v",
        "libx264",
        "-crf",
        "23",
        "-preset",
        "veryfast",
        output,
    ]

    subprocess.run(command, check=True)
    print(f"✅ Output saved to {output}")

assert len(ls_dirs) == 2, "Expected exactly two subdirectories in workdir."

def get_csv_files(dir_path):
    csv_files = fs.filter_files_by_extension(dir_path, ext=".csv")
    csv_files = [f for f in csv_files if f.endswith("_results.csv")]
    return csv_files

out_stack_dir = os.path.join(workdir, "all_stacked")
os.makedirs(out_stack_dir, exist_ok=True)

csvfiles_notemp = get_csv_files(ls_dirs[0])
csvfiles_temp = get_csv_files(ls_dirs[1])
for f1, f2 in tqdm(list(zip(csvfiles_notemp, csvfiles_temp))):
    # f1name = fs.get_file_name(f1, split_file_ext=True)[0]
    # f1name_target = f1name.replace("_results", "_notemp")
    # f2name = fs.get_file_name(f2, split_file_ext=True)[0]
    # f2name_target = f2name.replace("_results", "_temp")
    # f1_target = os.path.join(workdir, f"{f1name_target}.csv")
    # f2_target = os.path.join(workdir, f"{f2name_target}.csv")
    # fs.copy_file(f1, f1_target)
    # fs.copy_file(f2, f2_target)
    df1 = pd.read_csv(
        f1,
        sep=";",
        encoding="utf-8",
        dtype={"pred_label": str, "elapsed_time": float},
        keep_default_na=False,
    )
    df2 = pd.read_csv(
        f2,
        sep=";",
        encoding="utf-8",
        dtype={"pred_label": str, "elapsed_time": float},
        keep_default_na=False,
    )
    # replace all "skipped" to "None" string in df2
    df2['pred_label'] = df2['pred_label'].replace("skipped", "None")
    df1 = df1[["video", "frame_idx", "class_names", "probs", "pred_label"]]
    df2 = df2[['video', 'frame_idx', 'class_names', 'probs', 'pred_label']]
    df1.rename(columns={"probs": "probs_no_temp", "pred_label": "pred_label_no_temp"}, inplace=True)
    df2.rename(columns={"probs": "probs_temp", "pred_label": "pred_label_temp"}, inplace=True)
    df_merged = pd.merge(df1, df2, on=["video", "frame_idx", "class_names"])
    df_merged['pred_diff'] = df_merged.apply(lambda row: row['pred_label_no_temp'] != row['pred_label_temp'], axis=1)
    df_merged = df_merged[
        [
            "video",
            "frame_idx",
            "pred_diff",
            "pred_label_no_temp",
            "pred_label_temp",
            "class_names",
            "probs_no_temp",
            "probs_temp",
        ]
    ]
    # ! quick see by pred_diff
    df_merged = df_merged.sort_values(by=["pred_diff",'frame_idx'], ascending=[False, True])
    vname = fs.get_file_name(f1, split_file_ext=True)[0].split("_")[0]
    output_csv = os.path.join(workdir, f"{vname}.csv")
    df_merged.to_csv(output_csv, sep=';', encoding='utf-8', index=False)

    def get_out_videos(dir_path):
        videos = fs.filter_files_by_extension(dir_path, ext=".mp4")
        videos = [v for v in videos if v.endswith("_out.mp4")]
        return videos
    videos1 = get_out_videos(ls_dirs[0])
    videos2 = get_out_videos(ls_dirs[1])
    pprint(f"Found {len(videos1)} videos in {ls_dirs[0]}")
    pprint(f"Found {len(videos2)} videos in {ls_dirs[1]}")

    pair_ls_video = list(zip(videos1, videos2))
    for v1, v2 in tqdm(pair_ls_video):
        fname = fs.get_file_name(v1, split_file_ext=True)[0]
        vname = fname.split("_")[0]
        output_path = os.path.join(out_stack_dir, f"{vname}.mp4")
        if os.path.exists(output_path):
            pprint(f"Output {output_path} already exists, skip.")
            continue
        vstack(v1, v2, output=output_path)
pprint("Done")

  0%|          | 0/70 [00:00<?, ?it/s]

'Found 70 videos in ./zout/perf/__analyze/MainPC__ds_DFire__mt_no_temp__md_hgnetv2_b5.ssld_stage2_ft_in1k_360x640__20250921.182558'

'Found 70 videos in ./zout/perf/__analyze/MainPC__ds_DFire__mt_temp_stabilize__md_hgnetv2_b5.ssld_stage2_ft_in1k_360x640__20250921.182949'

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

✅ Output saved to ./zout/perf/__analyze/all_stacked/FP11.mp4


...

'Output ./zout/perf/__analyze/all_stacked/VP7.mp4 already exists, skip.'

'Output ./zout/perf/__analyze/all_stacked/VP8.mp4 already exists, skip.'

'Output ./zout/perf/__analyze/all_stacked/VP9.mp4 already exists, skip.'

100%|██████████| 70/70 [02:36<00:00,  2.24s/it]


'Done'

In [ ]:
wrong_df = pd.read_csv(os.path.join(workdir, "compare_wrong.csv"), sep=';', encoding='utf-8')
csv_file = wrong_df['video_name'].tolist()
wrong_dir = os.path.join(workdir, "wrong_videos")
pprint(wrong_dir)
os.makedirs(wrong_dir, exist_ok=True)
for v in csv_file:
    src_path = os.path.join(out_stack_dir, v)
    vname = fs.get_file_name(src_path, split_file_ext=True)[0]
    # pprint(src_path)
    src_csv_path = os.path.join(workdir, f"{vname}.csv")
    if os.path.exists(src_path):
        dst_path = os.path.join(wrong_dir, f"{vname}.mp4")
        fs.copy_file(src_path, dst_path)
        fs.copy_file(src_csv_path, os.path.join(wrong_dir, f"{vname}.csv"))

'./zout/perf/__analyze/wrong_videos'

In [5]:
from halib.research.perftb import *
from halib.research.perfcalc import PerfCalc

pertb = PerfCalc.gen_perf_report_for_multip_exps(
    indir=r"/mnt/e/NextCloud/paper2_main/zout/perf/__analyze"
)
pertb.plot("./zout/perf/summary.png")

/mnt/e/NextCloud/paper2_main/.venv/lib/python3.11/site-packages/halib/research/perfcalc.py:215: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, temp_df], ignore_index=True)


'------ Final DataFrame Columns ------'

+----+------------------------------------------+-----------+-------------------+-------------------+--------------------+-----------------------+---------------------------------+--------------+
|    | experiment                               | dataset   |   metric_accuracy |   metric_f1_score |   metric_precision |   metric_recall (TPR) |   metric_FPR (False Alarm Rate) |   metric_FPS |
+====+==========================================+===========+===================+===================+====================+=======================+=================================+==============+
|  0 | MainPC__ds_DFire__mt_no_temp__md_hgnetv2 | DFire     |          0.471429 |          0.554217 |           0.479167 |              0.657143 |                        0.714286 |      28.7557 |
|    | _b5.ssld_stage2_ft_in1k_360x640__2025092 |           |                   |                   |                    |                       |                                 |              |
|    | 1.182558_per_